This is a Personal AI Chatbot that:

- Reads your LinkedIn profile (via PDF)
- Uses OpenAI to understand and reason about it
- Lets users chat with it through a Gradio UI

In [1]:
# Import the necessary libraries
from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader
import gradio as gr

In [2]:
load_dotenv()  # Load environment variables from .env file
openai = OpenAI()  # Initialize the OpenAI client

In [3]:
reader = PdfReader("/workspaces/ai-agents-lab/1_foundations/me/Profile.pdf")
profile = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        profile += text

In [4]:
print(profile)

   
Contact
gbe01nga@gmail.com
www.linkedin.com/in/gbenga-
ojo-21767430a  (LinkedIn)
github.com/Charlie-Charlie01
(Personal)
Top Skills
Artificial Intelligence (AI)
Machine Learning
AutomationGbenga Ojo
AI/ML | Wordpress Developer | Web Automation Specialist | Building
Intelligent Web Solutions
Lagos State, Nigeria
Experience
Gidi Real Estate Investment Limited 
2 years 3 months
IT Specialist
May 2024 - Present  (2 years)
Lagos State, Nigeria
I am a passionate Web Developer with extensive experience in WordPress,
focusing on building engaging and accessible websites. Recently, I've been
diving into AI/ML and AI-driven automation, building intelligent systems
that streamline web workflows. I'm continuously learning tools like Python,
TensorFlow, and LangChain.
IT Specialist
February 2024 - Present  (2 years 3 months)
Lagos State, Nigeria
Gidi Real Estate Investment Limited 
Account Officer
 - June 2025 
Education
University of Ilorin, Nigeria
Bachelor of Technology - BTech, Building Con

In [5]:
# Read the summary from the text file
with open("/workspaces/ai-agents-lab/1_foundations/me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
# Set the name variable
name = "Ojo Gbenga Charles"

In [7]:
# Create the system prompt for the chatbot
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{profile}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [8]:
# Define the chat function that will be used to generate responses from the chatbot
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content 

In [9]:
# Launch the Gradio interface for the chatbot
gr.ChatInterface(chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7860


* Running on public URL: https://bc5b0c871d7ef6aa23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Using Gemini to Evaluate GPT-4's Responses - A Multi-LLM Pipeline

This is to help us to...
- Be able to ask an LLM to evaluate an answer
- Be able to rerun if the answer fails evaluation
- Put this together into 1 workflow

All without any Agentic framework

In [10]:
# Create a pydantic model for the Evaluation
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptible: bool
    feedback: str

In [11]:
# Create a system prompt for the evaluation agent
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{profile}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [12]:
# This function builds a prompt for an AI evaluator.
# Essentially an LLM that judges whether another AI agent's response was good or not.
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here is the conversation between the User and the Agent:\n\n{history}\n\n"
    user_prompt += f"Here is the latest message from the User:\n\n{message}\n\n"
    user_prompt += f"Here is the latest response from the Agent:\n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt


In [13]:
import os

openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

- We just wrapped Google's Gemini API using OpenAI's Python client.

In [14]:
# This function uses Gemini to evaluate an agent's reply and return a structured result.
# Building directly on the pieces you showed earlier.
def evaluate(reply, message, history) -> Evaluation:
    
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.beta.chat.completions.parse(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [15]:
# Let's send a message to OpenAI's GPT-4o-mini model and get a response
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [16]:
reply

'As of now, I do not hold any patents. My primary focus has been on developing skills in AI/ML and web automation, and I have been applying these skills in my work, but I have not pursued any patent applications. If you have any specific questions regarding my projects or areas of expertise, feel free to ask!'

In [17]:
# Now let's evaluate the reply using our Gemini evaluator
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptible=True, feedback="The response is acceptable. It directly answers the user's question about whether the Agent holds a patent, and it maintains a professional tone. Additionally, the Agent opens the door for further engagement by inviting the user to ask more questions about projects or expertise, which encourages interaction and keeps the conversation engaging.")

In [18]:
# If the reply was not acceptable, we can use the feedback to improve it and try again 
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"\n\n## Previous answer rejected \n You just tried to reply, but the quality control rejected your reply \n"
    updated_system_prompt += f"## Your attempted answer: \n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection: \n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [20]:
# Let's say the feedback from the evaluator was that the reply was not in pig latin (which is a silly requirement we added just to see if the system can handle it). 
# We can then rerun the chat function with the updated system prompt that includes the feedback.
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
            it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [21]:
# Finally, let's launch the Gradio interface for the chatbot again, now with the evaluation and rerun mechanism in place.
gr.ChatInterface(chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://f498a2468e68f4852a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
